In [8]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import struct

def save_vectors_binary(path, items):
    """
    items = [
      {
        "id": int,
        "v_core": np.ndarray,
        "v_desc": np.ndarray
      }
    ]
    """
    with open(path, "wb") as f:
        f.write(struct.pack("i", len(items)))   # số item
        f.write(struct.pack("i", len(items[0]["v_core"])))  # dim

        for it in items:
            f.write(struct.pack("i", it["id"]))
            f.write(struct.pack("i", it["type"]))
            f.write(it["v_core"].astype("float32").tobytes())
            f.write(it["v_desc"].astype("float32").tobytes())

def main():
    model = SentenceTransformer(
        "intfloat/multilingual-e5-base"
    )
    data_names = ["drinks", "foods"]
    items = []

    for name in data_names:
        df = pd.read_json(f"Data/products_{name}.json")

        print("Đang tạo vectors...")
        
        vectors_search = []
        for i in range(df.shape[0]):
            vector = model.encode("passage: " + df.iloc[i]["name"], normalize_embeddings=True)
            vectors_search.append(vector)
            print(f"  v_desc: {i+1}/{df.shape[0]}", end="\r")
        print()

        vectors_desc = []
        for i in range(df.shape[0]):
            desc_text = f"""
            {df.iloc[i]["name"]}.
            Thành phần: {df.iloc[i]["ingredients"]}.
            """
            vector = model.encode("passage: " + desc_text, normalize_embeddings=True)
            vectors_desc.append(vector)
            print(f"  v_core: {i+1}/{df.shape[0]}", end="\r")

        for i in range(len(vectors_search)):
            items.append({
                "id": i,
                "type": 0 if name == "drinks" else 1,
                "v_core": np.array(vectors_search[i]),
                "v_desc": np.array(vectors_desc[i])
            })
        
    save_vectors_binary("Vector/vectors.bin", items)
    print(f"Đã lưu {len(items)} vectors vào vectors.bin")

if __name__ == "__main__":
    main()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 978.24it/s, Materializing param=pooler.dense.weight]                               
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Đang tạo vectors...
  v_desc: 21/21
Đang tạo vectors...
  v_desc: 54/54
Đã lưu 75 vectors vào vectors.bin


In [10]:
import struct
import numpy as np

def read_vectors_binary(path):
    """
    Đọc vectors từ file binary
    Trả về list of dict: [{"id": int,"type": int, "v_core": np.ndarray, "v_desc": np.ndarray}, ...]
    """
    items = []
    with open(path, "rb") as f:
        num_items = struct.unpack("i", f.read(4))[0]
        dim = struct.unpack("i", f.read(4))[0]
        
        print(f"Số lượng items: {num_items}")
        print(f"Dimension: {dim}")
        print("=" * 50)
        
        for _ in range(num_items):
            item_id = struct.unpack("i", f.read(4))[0]
            type = struct.unpack("i", f.read(4))[0]
            v_core = np.frombuffer(f.read(dim * 4), dtype=np.float32)
            v_desc = np.frombuffer(f.read(dim * 4), dtype=np.float32)
            
            items.append({
                "id": item_id,
                "type": type,
                "v_core": v_core,
                "v_desc": v_desc
            })
    
    return items

items = read_vectors_binary("Vector/vectors.bin")

print("\n3 items đầu tiên:")
for item in items[:3]:
    print(f"\nID: {item['id']}")
    print(f"Type: {item['type']}")
    print(f"v_core shape: {item['v_core'].shape}, 5 giá trị đầu: {item['v_core'][:5]}")
    print(f"v_desc shape: {item['v_desc'].shape}, 5 giá trị đầu: {item['v_desc'][:5]}")

print("\n" + "=" * 50)
print("Kiểm tra normalization:")
print(f"Norm v_core[0]: {np.linalg.norm(items[0]['v_core']):.4f}")
print(f"Norm v_desc[0]: {np.linalg.norm(items[0]['v_desc']):.4f}")


Số lượng items: 75
Dimension: 768

3 items đầu tiên:

ID: 0
Type: 0
v_core shape: (768,), 5 giá trị đầu: [ 0.00089174  0.01185798 -0.02720548  0.02249287  0.01283867]
v_desc shape: (768,), 5 giá trị đầu: [ 0.01538255  0.04750726 -0.03040973  0.0449078   0.0079702 ]

ID: 1
Type: 0
v_core shape: (768,), 5 giá trị đầu: [ 0.00722057  0.02280374 -0.02039822  0.02230576  0.0116749 ]
v_desc shape: (768,), 5 giá trị đầu: [ 0.01205113  0.03723153 -0.01383683  0.0428235   0.00915587]

ID: 2
Type: 0
v_core shape: (768,), 5 giá trị đầu: [ 0.00529493  0.00430797 -0.02440077  0.0279658   0.00705659]
v_desc shape: (768,), 5 giá trị đầu: [-0.00148888  0.04922736 -0.03230833  0.05253465  0.02468073]

Kiểm tra normalization:
Norm v_core[0]: 1.0000
Norm v_desc[0]: 1.0000
